<a href="https://colab.research.google.com/github/Abhilash2240/Stock_Price_Prediction/blob/main/Stock_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Stock Price Prediction Project using Time Series Models
# Install and import necessary libraries
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn statsmodels tensorflow keras plotly openai streamlit flask -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
print("\n" + "="*60)
print("Libraries imported successfully!")
print("Real-time Financial Data Analysis and Forecasting")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 139.5 MB/s eta 0:00:00

Libraries imported successfully!
Real-time Financial Data Analysis and Forecasting


In [ ]:
# =====================================================
# STEP 2: LOAD REAL-TIME STOCK DATA
# =====================================================

print("\nLoading stock data...")

# Define stock ticker and date range
stock_ticker = 'AAPL'  # Apple Inc.
start_date = '2020-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

# Download stock data
df = yf.download(stock_ticker, start=start_date, end=end_date)

print(f"\nStock data loaded for {stock_ticker}")
print(f"Date range: {start_date} to {end_date}")
print(f"Total records: {len(df)}")
print("\nFirst 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())# =====================================================
# STEP 2: DOWNLOAD REAL-TIME STOCK DATA
# =====================================================

ticker = 'AAPL'
start_date = '2020-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

print(f"\nDownloading {ticker} stock data from {start_date} to {end_date}...")
df = yf.download(ticker, start=start_date, end=end_date)

print("\n" + "="*60)
print(f"✓ Downloaded {len(df)} days of stock data for {ticker}")
print("="*60)
print("\nFirst 5 rows:")
print(df.head())
print("\nData shape:", df.shape)
print("\nColumn names:", df.columns.tolist())


Loading stock data...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Stock data loaded for AAPL
Date range: 2020-01-01 to 2025-10-19
Total records: 1457

First 5 rows:
Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.538528  72.598907  71.292319  71.545905  135480400
2020-01-03  71.833313  72.594079  71.608707  71.765690  146322800
2020-01-06  72.405678  72.444321  70.703012  70.954188  118387200
2020-01-07  72.065170  72.671364  71.845392  72.415360  108872000
2020-01-08  73.224396  73.526287  71.768071  71.768071  132079200

Last 5 rows:
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-10-13  247.660004  249.690002  245.559998  249.380005  38142900
2025-10-14  247.770004  248.850006  244.699997  246.600006  35478000
2025-10-15  2

In [ ]:
# =====================================================
# STEP 4: DATA PREPROCESSING & VISUALIZATION
# =====================================================
"""
Preprocess stock data and create comprehensive visualizations:
1. Handle missing values
2. Feature engineering (moving averages, returns)
3. Interactive price charts
4. Technical indicators
5. Correlation analysis
"""

# Handle missing values
if df.isnull().sum().sum() > 0:
    df = df.fillna(method='ffill').fillna(method='bfill')
    print("Missing values filled")

# Flatten multi-level columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Feature Engineering: Add technical indicators
df['Daily_Return'] = df['Close'].pct_change() * 100  # Daily returns in %
df['MA_7'] = df['Close'].rolling(window=7).mean()  # 7-day moving average
df['MA_30'] = df['Close'].rolling(window=30).mean()  # 30-day moving average
df['MA_90'] = df['Close'].rolling(window=90).mean()  # 90-day moving average
df['Volatility'] = df['Daily_Return'].rolling(window=30).std()  # 30-day volatility
df['High_Low_Range'] = df['High'] - df['Low']  # Daily price range

print("\n✓ Technical indicators added:")
print("  - Daily Returns")
print("  - Moving Averages (7, 30, 90 days)")
print("  - Volatility (30-day rolling std)")
print("  - High-Low Range")
print(f"\nUpdated data shape: {df.shape}")
print(f"\nLatest data (last 5 rows with indicators):")
print(df[['Close', 'Daily_Return', 'MA_7', 'MA_30', 'Volatility']].tail())

# Create comprehensive visualizations
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Stock Price with Moving Averages', 'Daily Returns (%)', 'Trading Volume'),
    vertical_spacing=0.1,
    row_heights=[0.5, 0.25, 0.25]
)

# Plot 1: Price with moving averages
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close Price', line=dict(color='blue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MA_7'], name='MA 7', line=dict(color='orange', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MA_30'], name='MA 30', line=dict(color='green', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MA_90'], name='MA 90', line=dict(color='red', width=1)), row=1, col=1)

# Plot 2: Daily returns
fig.add_trace(go.Scatter(x=df.index, y=df['Daily_Return'], name='Daily Return', fill='tozeroy', line=dict(color='purple', width=1)), row=2, col=1)

# Plot 3: Volume
fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color='lightblue'), row=3, col=1)

fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Return (%)", row=2, col=1)
fig.update_yaxes(title_text="Volume", row=3, col=1)

fig.update_layout(height=900, title_text=f"{stock_ticker} Stock Analysis Dashboard", showlegend=True)
fig.show()

print("\n✓ Visualizations created successfully!")

Missing values filled

DATA PREPROCESSING

✓ Technical indicators added:
  - Daily Returns
  - Moving Averages (7, 30, 90 days)
  - Volatility (30-day rolling std)
  - High-Low Range

Updated data shape: (1457, 11)

Latest data (last 5 rows with indicators):
Price            Close  Daily_Return        MA_7       MA_30  Volatility
Date                                                                    
2025-10-13  247.660004      0.974436  253.745714  246.033666    1.710763
2025-10-14  247.770004      0.044416  252.281431  246.635333    1.694315
2025-10-15  249.339996      0.633649  251.231430  246.997666    1.559159
2025-10-16  247.449997     -0.758001  249.941428  247.253333    1.566168
2025-10-17  252.289993      1.955949  249.117142  247.673333    1.601289



✓ Visualizations created successfully!


In [ ]:
# =====================================================
# STEP 5: TIME SERIES MODELING - ARIMA
# =====================================================
"""
ARIMA (AutoRegressive Integrated Moving Average) Model:
- Time series forecasting for stock prices
- Stationarity test and differencing
- Model training and evaluation
- Future price predictions
"""

# Flatten multi-level columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print("\n" + "="*60)
print("ARIMA TIME SERIES MODELING")
print("="*60)

# Prepare data for ARIMA (use Close price)
close_prices = df['Close'].dropna().values

# Test for stationarity using Augmented Dickey-Fuller test
print("\nTesting for stationarity...")
result = adfuller(close_prices)
print(f"ADF Statistic: {result[0]:.6f}")
print(f"p-value: {result[1]:.6f}")
if result[1] <= 0.05:
    print("✓ Data is stationary (p-value <= 0.05)")
else:
    print("⚠ Data is non-stationary (p-value > 0.05)")

# Split data into train and test sets (80-20 split)
train_size = int(len(close_prices) * 0.8)
train_data = close_prices[:train_size]
test_data = close_prices[train_size:]

print(f"\nTraining samples: {len(train_data)}")
print(f"Testing samples: {len(test_data)}")

# Train ARIMA model
print("\nTraining ARIMA(5,1,0) model...")
arima_model = ARIMA(train_data, order=(5, 1, 0))
arima_fitted = arima_model.fit()
print("✓ ARIMA model trained successfully!")

# Make predictions
arima_predictions = arima_fitted.forecast(steps=len(test_data))

# Calculate evaluation metrics
mse = mean_squared_error(test_data, arima_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_data, arima_predictions)
r2 = r2_score(test_data, arima_predictions)

print("\n" + "="*60)
print("ARIMA MODEL EVALUATION")
print("="*60)
print(f"RMSE: ${rmse:.2f}")
print(f"MAE: ${mae:.2f}")
print(f"R² Score: {r2:.4f}")

# Forecast future prices (30 days)
future_days = 30
future_predictions = arima_fitted.forecast(steps=len(test_data) + future_days)
future_forecast = future_predictions[-future_days:]

print(f"\nForecast (next {future_days} days):")
print(f"Day 1: ${future_forecast[0]:.2f}")
print(f"Day 15: ${future_forecast[14]:.2f}")
print(f"Day {future_days}: ${future_forecast[-1]:.2f}")

# Visualize results
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(train_data))), y=train_data, name='Training', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=list(range(len(train_data), len(train_data)+len(test_data))), y=test_data, name='Actual Test', line=dict(color='green')))
fig.add_trace(go.Scatter(x=list(range(len(train_data), len(train_data)+len(test_data))), y=arima_predictions, name='ARIMA Predictions', line=dict(color='red', dash='dash')))
fig.add_trace(go.Scatter(x=list(range(len(train_data)+len(test_data), len(train_data)+len(test_data)+future_days)), y=future_forecast, name='Future Forecast', line=dict(color='orange', dash='dot')))
fig.update_layout(title='ARIMA Stock Price Prediction', xaxis_title='Time', yaxis_title='Price ($)', height=500)
fig.show()

print("\n✓ ARIMA modeling completed!")


ARIMA TIME SERIES MODELING

Testing for stationarity...
ADF Statistic: -1.202198
p-value: 0.672649
⚠ Data is non-stationary (p-value > 0.05)

Training samples: 1165
Testing samples: 292

Training ARIMA(5,1,0) model...
✓ ARIMA model trained successfully!

ARIMA MODEL EVALUATION
RMSE: $17.10
MAE: $13.78
R² Score: -0.0002

Forecast (next 30 days):
Day 1: $224.93
Day 15: $224.93
Day 30: $224.93



✓ ARIMA modeling completed!


In [ ]:
# =====================================================
# STEP 5: LSTM DEEP LEARNING MODEL
# =====================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

print("\n" + "="*60)
print("BUILDING LSTM DEEP LEARNING MODEL")
print("="*60)

# Prepare data for LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df['Close'].values.reshape(-1,1))

# Create sequences
def create_sequences(data, seq_length=60):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, seq_length=60)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape for LSTM
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Build LSTM model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
print("\nTraining LSTM model...")
history = model.fit(X_train, y_train, batch_size=32, epochs=20, validation_split=0.1, verbose=0)

# Make predictions
lstm_predictions = model.predict(X_test)
lstm_predictions = scaler.inverse_transform(lstm_predictions)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluate
rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, lstm_predictions))
mae_lstm = mean_absolute_error(y_test_actual, lstm_predictions)
r2_lstm = r2_score(y_test_actual, lstm_predictions)

print("\n✓ LSTM Model Completed")
print(f"RMSE: ${rmse_lstm:.2f}")
print(f"MAE: ${mae_lstm:.2f}")
print(f"R2 Score: {r2_lstm:.4f}")


BUILDING LSTM DEEP LEARNING MODEL

X_train shape: (1117, 60, 1)
X_test shape: (280, 60, 1)

Training LSTM model...
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

✓ LSTM Model Completed
RMSE: $8.45
MAE: $6.60
R2 Score: 0.7652


In [ ]:
# ============================================================
# STEP 1: API AUTHENTICATION AND CONFIGURATION
# ============================================================
print("\nStep 1: Configuring API Authentication...")
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_AI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✓ API Key loaded from Colab Secrets")
    print("✓ Gemini API configured successfully\n")
except Exception as e:
    print("⚠️ API Key not found in Colab Secrets")
    print("\n🔑 To add your API key:")
    print("   1. Get API key: https://aistudio.google.com/apikey")
    print("   2. Click 'Secrets' (🔑) in left sidebar")
    print("   3. Add: Name='GOOGLE_AI_API_KEY', Value='your-key'")
    print("   4. Enable notebook access")
    print("   5. Re-run this cell\n")


Step 1: Configuring API Authentication...
⚠️ API Key not found in Colab Secrets

🔑 To add your API key:
   1. Get API key: https://aistudio.google.com/apikey
   2. Click 'Secrets' (🔑) in left sidebar
   3. Add: Name='GOOGLE_AI_API_KEY', Value='your-key'
   4. Enable notebook access
   5. Re-run this cell



In [ ]:
# ============================================================
# GOOGLE AI STUDIO (GEMINI) API INTEGRATION FOR STOCK PREDICTION
# ============================================================
# This section integrates Google's Gemini API for:
# 1. API Authentication and Setup
# 2. Feature Generation using AI
# 3. Market Sentiment Analysis
# 4. Enhanced Predictions with AI Insights
# ============================================================

# Install Google Generative AI SDK
!pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata
import json
import time

print("\n" + "="*60)
print("📡 GOOGLE AI STUDIO (GEMINI) API INTEGRATION")
print("="*60)


📡 GOOGLE AI STUDIO (GEMINI) API INTEGRATION


In [ ]:
# =====================================================
# STEP 6: GEMINI API INTEGRATION FOR ENHANCED PREDICTIONS
# =====================================================
print("\n" + "="*60)
print("GEMINI API - MARKET SENTIMENT & FEATURE ENHANCEMENT")
print("="*60)

try:
    # Configure Gemini API
    GOOGLE_API_KEY = userdata.get('GOOGLE_AI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel('gemini-pro')
    print("\n✓ Gemini API configured successfully")

    # Generate AI-powered market sentiment analysis
    latest_price = df['Close'].iloc[-1]
    latest_ma7 = df['MA_7'].iloc[-1]
    latest_volatility = df['Volatility'].iloc[-1]

    prompt = f"""Analyze AAPL stock: Current price ${latest_price:.2f}, 7-day MA ${latest_ma7:.2f}, Volatility {latest_volatility:.2f}.
    Provide a brief sentiment (50 words): Bullish/Bearish/Neutral and key factors."""

    response = model.generate_content(prompt)
    print("\n✓ AI-Powered Market Sentiment Analysis:")
    print(response.text)

except Exception as e:
    print(f"\n⚠️ Gemini API not configured: {str(e)}")
    print("Note: Add GOOGLE_AI_API_KEY to Colab Secrets to enable AI features")


GEMINI API - MARKET SENTIMENT & FEATURE ENHANCEMENT

⚠️ Gemini API not configured: Secret GOOGLE_AI_API_KEY does not exist.
Note: Add GOOGLE_AI_API_KEY to Colab Secrets to enable AI features


In [ ]:
# STEP 7: WEB INTERFACE WITH STREAMLIT (Deploy)
import subprocess, sys, time, os

# Ensure deps and write app file already created in Cell 9
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit', 'pyngrok'], check=False)

# Import necessary libraries
from google.colab import userdata
from pyngrok import ngrok

# Get ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
print(f"NGROK_AUTH_TOKEN retrieved: {'Yes' if NGROK_AUTH_TOKEN else 'No'}")
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("Ngrok authtoken loaded from Colab Secrets.")
else:
    print("Ngrok authtoken not found in Colab Secrets. Please add it to use ngrok.")
    print("\n🔑 To add your ngrok authtoken:")
    print("   1. Get authtoken: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   2. Click 'Secrets' (🔑) in left sidebar")
    print("   3. Add: Name='NGROK_AUTH_TOKEN', Value='your-token'")
    print("   4. Enable notebook access")
    print("   5. Re-run this cell\n")


# Start Streamlit app in background
cmd = "streamlit run stock_predictor_app.py --server.port 8501 --browser.gatherUsageStats false"
print("Starting Streamlit server...")
proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Expose via ngrok
try:
    public_url = ngrok.connect(8501)
    print("Streamlit public URL:", public_url)

    # Wait a bit for server to be ready
    for i in range(10):
        time.sleep(2)
        print("Waiting for server to be ready...", i+1)

    print("✅ Deployed. Access the app at:", public_url)

except Exception as e:
    print(f"An error occurred while trying to connect with ngrok: {e}")
    print("Please ensure your ngrok authtoken is correctly set in Colab secrets.")

NGROK_AUTH_TOKEN retrieved: Yes
Ngrok authtoken loaded from Colab Secrets.
Starting Streamlit server...
Streamlit public URL: NgrokTunnel: "https://remuneratively-overrash-belia.ngrok-free.dev" -> "http://localhost:8501"
Waiting for server to be ready... 1
Waiting for server to be ready... 2


Waiting for server to be ready... 3


Waiting for server to be ready... 4
Waiting for server to be ready... 5
Waiting for server to be ready... 6
Waiting for server to be ready... 7
Waiting for server to be ready... 8
Waiting for server to be ready... 9
Waiting for server to be ready... 10
✅ Deployed. Access the app at: NgrokTunnel: "https://remuneratively-overrash-belia.ngrok-free.dev" -> "http://localhost:8501"
